In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler


In [ ]:
#Process data
%run data_handler.ipynb

In [ ]:
X = inputs
y = outputs

In [ ]:
%run ../common_functions.ipynb
# Initialize result collection
query_results = []

In [ ]:
param_grid = [
    {'method': 'PI', 'xi': 0.01},
    {'method': 'PI', 'xi': 0.05},
    {'method': 'PI', 'xi': 0.1},
    {'method': 'UCB', 'kappa': 1.0},
    {'method': 'UCB', 'kappa': 2.0},
    {'method': 'UCB', 'kappa': 3.0},
    {'method': 'EI', 'xi': 0.01},
    {'method': 'EI', 'xi': 0.05},
    {'method': 'EI', 'xi': 0.1},
]

# --- Shared precomputation ---
print("Starting acquisition sweep...")

# Shared config
apply_scaling = True
grid_size = 100
dimension = X.shape[1] - 1
filter_mode = 'gp'

print("Number of features:", X.shape[1])
y_scaled = preprocess_y(y, apply_scaling)
bounds = compute_bounds(X)
X_grid = create_nd_grid(bounds, grid_size, dimension)
X_grid_filtered = filter_grid(X, y_scaled, X_grid, filter_mode)
gp_model = train_gp(X, y_scaled)

mean, std = gp_model.predict(X_grid_filtered, return_std=True)
print("GP prediction stats:")
print("  Mean shape:", mean.shape)
print("  Std shape :", std.shape)

# --- Parameter sweep ---
for params in param_grid:
    method = params['method']
    xi = params.get('xi', 0.1)
    kappa = params.get('kappa', 2.0)
    label_param = xi if method in ['PI', 'EI'] else kappa

    next_point, acq_vals, _, _ = select_next_query_from_gp(
        X_grid=X_grid_filtered, y=y_scaled, method=method, xi=xi, kappa=kappa,
        mean=mean, std=std  # reuse cached predictions
    )
    score = acq_vals[np.argmax(acq_vals)]
    print(f"[{method} | {label_param}] → Next query point: {next_point}, Score: {score:.4f}")
    log_query_candidate(method, label_param, next_point, score)

In [ ]:
find_best_candidate(query_results)

In [ ]:
#add points to dataframe
add_points_to_df(df_sorted, query_results)
pd.set_option('display.max_rows', None)
print(df_sorted)

In [ ]:
plot_output_points(df_sorted, scale_factor=20.0, yield_scale_factor=1.0)